# Búsqueda rápida del arreglo de loops (filtro + retomar en el loop), Colab L4

La búsqueda larga prueba 12 arreglos resolviendo problemas completos (horas). Esta versión:

1. **Filtro (2 min):** califica los 12 con una sola pasada y **tira los 6 peores**.
2. **Retomar en el loop:** toma respuestas donde el modelo base se quedó en loop, las corta **justo antes** del loop y deja que cada arreglo siga desde ahí (1,024 tokens). Mide si sale del loop y si llega a la respuesta correcta. Además revisa que el arreglo no eche a perder problemas que el modelo ya resolvía bien.
3. **Elige** el que estime más aciertos.

**Validación (regla fija en GitHub antes de correr):** sobre los datos del #13 (Qwen3.5-0.8B, `wsi_runs/discover_blind`), donde la búsqueda larga ya midió a cada arreglo. Predicción: su elección queda entre las 3 mejores de la ronda 1 (empates cuentan). El 4B no va: en GSM8K casi no hace loops (1/200 en el examen).

Primero corre una **prueba de humo** (~5 min, pocos datos) para que si algo truena, truene ahí. Solo lee la carpeta del #13; trabaja en copias. **Uso:** `L4` → `Run all` → acepta Drive. ~20 min. Resultado en `MyDrive/wsi_runs/FINALES/quick_discover.json`.

In [ ]:
RUNS = [("Qwen/Qwen3.5-0.8B", "discover_blind")]
REPO = "/content/wsi"
Q = f"{REPO}/scripts/quick_discover.py"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm
!pip install -q flash-linear-attention || echo "fla opcional"
import vllm, transformers; print("vllm", vllm.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, shutil, json, subprocess, sys, datetime
from google.colab import drive
drive.mount("/content/drive")
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
os.makedirs(FINALES, exist_ok=True)
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git pull -q
assert os.path.exists(Q), f"{Q} no está en GitHub todavía: falta hacer push"
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
print("commit", COMMIT)

In [ ]:
from huggingface_hub import snapshot_download
NEED = ["config.json", "requests_elicit.jsonl", "outputs_elicit.jsonl", "diagnosis_blind.json", "blind_directions.npz"]
results = {}
for model, src_name in RUNS:
    tag = model.split("/")[-1].lower().replace(".", "")
    src = f"/content/drive/MyDrive/wsi_runs/{src_name}"
    w = f"/content/quick_{tag}"
    keep = f"/content/drive/MyDrive/wsi_runs/quick_{tag}"   # progreso: si se desconecta, sigue desde aquí
    os.makedirs(w, exist_ok=True); os.makedirs(keep, exist_ok=True)
    for f in NEED + ["search.json"]:
        if os.path.exists(f"{src}/{f}") and not os.path.exists(f"{w}/{f}"):
            shutil.copy2(f"{src}/{f}", w)              # solo lectura de la carpeta original
    for f in os.listdir(keep):
        if not os.path.exists(f"{w}/{f}"):
            shutil.copy2(f"{keep}/{f}", w)
    missing = [f for f in NEED if not os.path.exists(f"{w}/{f}")]
    if missing:
        print(f"SALTO {model}: faltan en {src}: {missing}"); continue
    base_dir = json.load(open(f"{w}/config.json"))["base_dir"]
    snapshot_download(model, local_dir=base_dir)        # donde la corrida original lo tenía
    print(f"===== {model}: prueba de humo =====", flush=True)
    ws = w + "_smoke"
    shutil.rmtree(ws, ignore_errors=True); os.makedirs(ws)
    for f in NEED + ["search.json"]:                    # solo los datos de entrada, nada de la corrida completa
        if os.path.exists(f"{w}/{f}"):
            shutil.copy2(f"{w}/{f}", ws)
    os.environ.pop("WSI_SYNC_DIR", None)
    r = subprocess.run([sys.executable, "-u", Q, "all", "--work", ws, "--smoke"], capture_output=True, text=True)
    print(r.stdout[-3000:]); print(r.stderr[-3000:])
    assert r.returncode == 0, "la prueba de humo falló: pégale el error a Claude"
    print("prueba de humo OK")
    os.environ["WSI_SYNC_DIR"] = keep
    print(f"===== {model} =====", flush=True)
    r = subprocess.run([sys.executable, "-u", Q, "all", "--work", w], capture_output=True, text=True)
    print(r.stdout[-8000:]); print(r.stderr[-4000:])
    if r.returncode == 0:
        results[model] = json.load(open(f"{w}/results_quick_discover.json"))
    shutil.rmtree(base_dir, ignore_errors=True)        # libera disco para el siguiente modelo

In [ ]:
for m, res in results.items():
    v = res.get("vs_search", {})
    print(f"{m}: elige {res['choice']} en {res.get('quick_min')} min (búsqueda larga: {v.get('search_min')} min)")
    print(f"   ronda 1 de su elección {v.get('r1_accuracy_of_choice')} vs 3.ª mejor {v.get('r1_third_best')} ->",
          "ACIERTO" if v.get("prediction_hit") else "FALLO", f"(al azar {v.get('chance')})")
out = {"runs": results, "_final": {"complete": len(results) == len(RUNS), "commit": COMMIT,
       "notebook": "quick_discover_colab.ipynb",
       "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}}
dst = f"{FINALES}/quick_discover.json"
with open(dst + ".tmp", "w") as f:
    json.dump(out, f, indent=1)
os.replace(dst + ".tmp", dst)
print("guardado en", dst)

In [ ]:
# Apagar el runtime: solo si el resultado completo ya está en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/quick_discover.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(90)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: falta algún modelo en FINALES.")